# Label Forge - Google Colab Training Notebook

This notebook trains YOLO models on Colab GPU and syncs metrics back to Label Forge.

In [ ]:
# @title Setup Parameters
JOB_ID = "job_id_here"  # @param {type:"string"}
DATASET_URL = "https://backend-public.example.com/datasets/xyz.zip"  # @param {type:"string"}
CALLBACK_URL = "https://backend-public.example.com/training/job_id/colab-callback"  # @param {type:"string"}
ARCHITECTURE = "yolov8s"  # @param ["yolov8n", "yolov8s", "yolov8m", "yolov8l", "yolov8x"]
EPOCHS = 50  # @param {type:"integer"}
IMAGE_SIZE = 640  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
LEARNING_RATE = 0.006  # @param {type:"number"}
PATIENCE = 12  # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25  # @param {type:"number"}

print(f"Job ID: {JOB_ID}")
print(f"Dataset URL: {DATASET_URL}")
print(f"Callback URL: {CALLBACK_URL}")
print(f"Training: {ARCHITECTURE}, {EPOCHS} epochs, image {IMAGE_SIZE}, batch {BATCH_SIZE}")


In [ ]:
# @title Validate Public URLs
from urllib.parse import urlparse

def is_local_url(url: str) -> bool:
    parsed = urlparse(url)
    host = parsed.hostname or ""
    return host in {"localhost", "127.0.0.1", "0.0.0.0"}

if is_local_url(DATASET_URL) or is_local_url(CALLBACK_URL):
    raise RuntimeError(
        "DATASET_URL and CALLBACK_URL must be public URLs. Set BACKEND_PUBLIC_URL on the backend (for example, a cloudflared https://xxxxx.trycloudflare.com URL) and reopen the Colab link."
    )

print("✓ Public URLs detected. Training can proceed.")

In [ ]:
# @title Install Dependencies
print("Installing dependencies...")
!pip install -q ultralytics torch torchvision requests gdown pyyaml

In [ ]:
# @title Check GPU
import torch
has_cuda = torch.cuda.is_available()
print(f"GPU Available: {has_cuda}")
if has_cuda:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# @title Download Dataset
import os
import io
import glob
import zipfile
import requests
import yaml

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def count_images(directory):
    if not os.path.isdir(directory):
        return 0
    return sum(
        1
        for name in os.listdir(directory)
        if name.lower().endswith(IMAGE_EXTENSIONS)
    )


def resolve_dataset_path(value, default_relative):
    value = value or default_relative
    if isinstance(value, list):
        value = value[0] if value else default_relative
    value = str(value)
    return value if os.path.isabs(value) else os.path.normpath(os.path.join(dataset_dir, value))

print("Downloading dataset...")
work_dir = f"/content/labelforge_train_{JOB_ID}"
dataset_dir = os.path.join(work_dir, "dataset")
os.makedirs(dataset_dir, exist_ok=True)

response = requests.get(DATASET_URL, stream=True, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall(dataset_dir)

print(f"✓ Dataset extracted to {dataset_dir}")
print(f"  Files: {os.listdir(dataset_dir)}")

data_yaml = os.path.join(dataset_dir, "data.yaml")
is_classification = not os.path.exists(data_yaml)

if is_classification:
    print("✓ Classification dataset detected (no data.yaml found).")
    train_images_dir = os.path.join(dataset_dir, "train")
    valid_images_dir = os.path.join(dataset_dir, "valid")
    test_images_dir = os.path.join(dataset_dir, "test")
    
    def count_class_images(directory):
        if not os.path.isdir(directory):
            return 0
        total = 0
        for root, dirs, files in os.walk(directory):
            for f in files:
                if f.lower().endswith(IMAGE_EXTENSIONS):
                    total += 1
        return total

    split_counts = {
        "train": count_class_images(train_images_dir),
        "valid": count_class_images(valid_images_dir),
        "test": count_class_images(test_images_dir),
    }
    data_config = {}
else:
    print("✓ Object detection dataset detected (data.yaml found).")
    with open(data_yaml, "r") as f:
        data_config = yaml.safe_load(f) or {}

    train_images_dir = resolve_dataset_path(data_config.get("train"), "train/images")
    valid_images_dir = resolve_dataset_path(data_config.get("val"), "valid/images")
    test_images_dir = resolve_dataset_path(data_config.get("test"), "test/images")

    split_counts = {
        "train": count_images(train_images_dir),
        "valid": count_images(valid_images_dir),
        "test": count_images(test_images_dir),
    }

if split_counts["train"] == 0:
    raise RuntimeError("No training images found. Create a dataset version with at least one train image.")

if not is_classification:
    if split_counts["valid"] == 0:
        print("⚠ No validation images found. Using train/images as YOLO val split for this run.")
        data_config["val"] = data_config.get("train", "./train/images")
        valid_images_dir = train_images_dir
        split_counts["valid"] = split_counts["train"]

    if split_counts["test"] == 0:
        print("⚠ No test images found. Using validation split as YOLO test split for this run.")
        data_config["test"] = data_config.get("val", data_config.get("train", "./train/images"))
        test_images_dir = valid_images_dir
        split_counts["test"] = split_counts["valid"]

    with open(data_yaml, "w") as f:
        yaml.safe_dump(data_config, f, sort_keys=False)
    print("✓ data.yaml ready")

print("  Split counts:", split_counts)
DATA_CONFIG = data_config
SPLIT_COUNTS = split_counts
TRAIN_IMAGES_DIR = train_images_dir
VALID_IMAGES_DIR = valid_images_dir
TEST_IMAGES_DIR = test_images_dir


In [ ]:
# @title YOLO Training
from ultralytics import YOLO
from datetime import datetime

print()
print("=" * 60)
print("STARTING YOLO TRAINING")
print("=" * 60)

architecture = ARCHITECTURE
image_size = int(IMAGE_SIZE)
batch_size = int(BATCH_SIZE)
epochs = int(EPOCHS)
learning_rate = float(LEARNING_RATE)
patience = int(PATIENCE)
confidence_threshold = float(CONFIDENCE_THRESHOLD)

runs_dir = os.path.join(work_dir, "runs")
has_cuda = bool(globals().get("has_cuda", torch.cuda.is_available()))
if not has_cuda:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

if has_cuda:
    device = 0
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory_gb >= 20:
        max_batch_by_gpu = 32
        training_workers = 8
    elif gpu_memory_gb >= 14:
        max_batch_by_gpu = 24
        training_workers = 6
    elif gpu_memory_gb >= 8:
        max_batch_by_gpu = 16
        training_workers = 4
    else:
        max_batch_by_gpu = 8
        training_workers = 2
else:
    device = "cpu"
    max_batch_by_gpu = 4
    training_workers = 2
    print("⚠ No GPU detected. Training will run on CPU and will be much slower.")

model_batch_limits = {
    "yolov8n": 32,
    "yolov8s": 24,
    "yolov8m": 12,
    "yolov8l": 6,
    "yolov8x": 4,
}
training_batch_size = max(1, min(batch_size, max_batch_by_gpu, model_batch_limits.get(architecture, 16)))

is_classification = not os.path.exists(os.path.join(dataset_dir, "data.yaml"))
if is_classification:
    model_name = f"{architecture}-cls.pt" if not architecture.endswith("-cls") else f"{architecture}.pt"
    data_arg = dataset_dir
else:
    model_name = f"{architecture}.pt"
    data_arg = data_yaml

print(f"Model: {model_name}")
print(f"Image Size: {image_size}")
print(f"Batch Size: {training_batch_size}")
print(f"Epochs: {epochs}")
print(f"Learning Rate: {learning_rate}")
print(f"Device: {device}")
print(f"Dataset Images: train={SPLIT_COUNTS['train']}, val={SPLIT_COUNTS['valid']}, test={SPLIT_COUNTS['test']}\n")

model = YOLO(model_name)
print(f"Training started at {datetime.now().isoformat()}")
train_results = model.train(
    data=data_arg,
    epochs=epochs,
    imgsz=image_size,
    batch=training_batch_size,
    lr0=learning_rate,
    patience=patience,
    project=runs_dir,
    name="train",
    exist_ok=True,
    device=device,
    workers=training_workers,
    plots=False,
    verbose=True,
)

print()
print(f"Training completed at {datetime.now().isoformat()}")
print("=" * 60)


In [ ]:
# @title Evaluate Model & Extract Metrics
print()
print("Evaluating model...")
if TRAIN_IMAGES_DIR == VALID_IMAGES_DIR:
    print("⚠ Validation is using train/images because this dataset version has no separate validation split.")

save_dir = getattr(train_results, "save_dir", None) or os.path.join(runs_dir, "train")
best_model_path = os.path.join(str(save_dir), "weights", "best.pt")

if not os.path.exists(best_model_path):
    raise RuntimeError(f"best.pt not found at {best_model_path}")

trained_model = YOLO(best_model_path)
if is_classification:
    validation = trained_model.val(data=dataset_dir, imgsz=image_size, verbose=False)
else:
    validation = trained_model.val(data=data_yaml, imgsz=image_size, verbose=False)
metrics = getattr(validation, "results_dict", {}) or {}

def get_metric_value(metrics_dict, candidates):
    for key in candidates:
        value = metrics_dict.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                pass
    return 0.0

if is_classification:
    map_score = get_metric_value(metrics, ["metrics/accuracy_top1"])
    precision = get_metric_value(metrics, ["metrics/accuracy_top5"])
    recall = 0.0
else:
    map_score = get_metric_value(metrics, ["metrics/mAP50-95(B)", "metrics/mAP50-95"])
    precision = get_metric_value(metrics, ["metrics/precision(B)", "metrics/precision"])
    recall = get_metric_value(metrics, ["metrics/recall(B)", "metrics/recall"])

print(f"mAP50-95 (or Top-1 Acc): {map_score:.4f}")
print(f"Precision (or Top-5 Acc): {precision:.4f}")
print(f"Recall: {recall:.4f}")

training_metrics = {
    "map_score": float(map_score),
    "precision": float(precision),
    "recall": float(recall),
    "epochs": epochs,
}


In [ ]:
# @title Generate Sample Predictions
import glob

print()
print("Generating sample predictions...")

sample_predictions = []
sample_images_dir = VALID_IMAGES_DIR if os.path.isdir(VALID_IMAGES_DIR) else TRAIN_IMAGES_DIR
sample_split_name = "validation" if sample_images_dir == VALID_IMAGES_DIR else "train"

image_paths = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
    if is_classification:
        pattern = os.path.join(sample_images_dir, "**", ext)
        image_paths.extend(glob.glob(pattern, recursive=True))
    else:
        pattern = os.path.join(sample_images_dir, ext)
        image_paths.extend(glob.glob(pattern))
    if len(image_paths) >= 6:
        break
image_paths = image_paths[:6]
print(f"Found {len(image_paths)} {sample_split_name} images for sample predictions")

if image_paths:
    names = getattr(trained_model, "names", {}) or {}
    results = trained_model.predict(image_paths, conf=confidence_threshold, verbose=False)
    for result in results:
        image_name = os.path.basename(getattr(result, "path", "sample.jpg"))
        if is_classification:
            probs = getattr(result, "probs", None)
            if probs is not None:
                top1_idx = int(probs.top1)
                top1_conf = float(probs.top1conf.item() if hasattr(probs.top1conf, "item") else probs.top1conf)
                sample_predictions.append({
                    "image_name": image_name,
                    "class_name": names.get(top1_idx, str(top1_idx)),
                    "confidence": round(top1_conf, 4),
                    "bbox": None
                })
        else:
            boxes = getattr(result, "boxes", None)
            if boxes is None:
                continue
            for box in boxes[:1]:
                cls_idx = int(box.cls[0].item())
                xywhn = box.xywhn[0].tolist()
                sample_predictions.append({
                    "image_name": image_name,
                    "class_name": names.get(cls_idx, str(cls_idx)),
                    "confidence": round(float(box.conf[0].item()), 4),
                    "bbox": {
                        "x": round(float(xywhn[0] - xywhn[2] / 2), 4),
                        "y": round(float(xywhn[1] - xywhn[3] / 2), 4),
                        "width": round(float(xywhn[2]), 4),
                        "height": round(float(xywhn[3]), 4),
                    },
                })
    print(f"✓ Generated {len(sample_predictions)} sample predictions")
else:
    print("No images found for sample predictions")

training_metrics["sample_predictions"] = sample_predictions[:6]


In [ ]:
# @title Upload best.pt through Label Forge backend
import requests
import os

print("Uploading best.pt through Label Forge backend...")
# Derive backend base URL from CALLBACK_URL (CALLBACK_URL = <backend>/api/training/{JOB_ID}/colab-callback)
backend_base = CALLBACK_URL.replace(f"/api/training/{JOB_ID}/colab-callback", "")
upload_endpoint = f"{backend_base}/api/training/{JOB_ID}/upload-artifact"

with open(best_model_path, 'rb') as f:
    files = {"file": ("best.pt", f, "application/octet-stream")}
    r = requests.post(upload_endpoint, files=files, timeout=300)
    try:
        r.raise_for_status()
    except Exception as e:
        raise RuntimeError(f"Upload failed: {e} - {r.status_code} {r.text}")
    data = r.json()

model_url = data.get("model_url")
artifact_key = data.get("artifact_key")
if not model_url or not artifact_key:
    raise RuntimeError("Invalid upload response from backend")

print("✓ Upload successful")
print(f"Model will be available as: {model_url}")


In [ ]:
# @title Send Results Back to Label Forge
import requests
import json

print("Sending results back to Label Forge...")
payload = {
    "status": "done",
    "metrics": {
        "map_score": training_metrics["map_score"],
        "precision": training_metrics["precision"],
        "recall": training_metrics["recall"],
        "epochs": training_metrics["epochs"],
    },
    "epochs": training_metrics["epochs"],
    "sample_predictions": training_metrics["sample_predictions"],
    "model_url": model_url,
}

print(json.dumps(payload, indent=2))
response = requests.post(CALLBACK_URL, json=payload, timeout=30)
print(f"Callback status: {response.status_code}")
response.raise_for_status()
print("✓ Results successfully sent to Label Forge")


## Training Complete

Results have been sent back to Label Forge. Check the training dashboard for metrics and predictions.